## Gold Star Schema

Build a single-date salary snapshot from validated Silver employee data. Tables are stored in Unity Catalog as Delta.

Source: `employees_pipeline.silver.employees_clean`  
Target: `employees_pipeline.gold`


In [0]:

from pyspark.sql import functions as F

silver_df = spark.table(
    "employees_pipeline.silver.employees_clean"
)

print("Silver records:", silver_df.count())


## 1. Employee dimension

Keep one row per employee with basic employee details.


In [0]:
dim_employee_df = (
    silver_df
    .select(
        "employee_id",
        "first_name",
        "last_name",
        "email",
        "hire_date"
    )
)
display(dim_employee_df)

In [0]:

print("Total employees:", dim_employee_df.count())

print(
    "Unique employees:",
    dim_employee_df.select("employee_id").distinct().count()
)

print(
    "NULL employee_id:",
    dim_employee_df.filter(F.col("employee_id").isNull()).count()
)


In [0]:

(
    dim_employee_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employees_pipeline.gold.dim_employee")
)

print("DIM_EMPLOYEE saved successfully")


## 2. Department dimension

The source has department IDs but no department names. The mapping below uses sample names for this project.


In [0]:

silver_df.select("department_id").distinct().orderBy("department_id").show()


In [0]:

departments_data = [
    (10, "IT"),
    (20, "HR"),
    (30, "Finance"),
    (40, "Marketing"),
    (50, "Operations")
]

dim_department_df = spark.createDataFrame(
    departments_data,
    ["department_id", "department_name"]
)

display(dim_department_df)


In [0]:

(
    dim_department_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employees_pipeline.gold.dim_department")
)

print("DIM_DEPARTMENT saved successfully")


## 3. Date dimension

Use one snapshot date for this run. This is not a historical salary dataset.


In [0]:

from datetime import date

snapshot_date = date.today()

print("Snapshot date:", snapshot_date)


In [0]:

dim_date_df = (
    spark.createDataFrame(
        [(snapshot_date,)],
        ["full_date"]
    )
    .withColumn(
        "date_id",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .select(
        "date_id",
        "full_date",
        "year",
        "month",
        "day"
    )
)

display(dim_date_df)


In [0]:

(
    dim_date_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employees_pipeline.gold.dim_date")
)

print("DIM_DATE saved successfully")


## 4. Salary fact

Grain: one employee per snapshot date. Each row stores a salary value and keys to the dimensions.


In [0]:

fact_employee_salary_df = (
    silver_df
    .select(
        "employee_id",
        "department_id",
        "salary"
    )
    .withColumn(
        "date_id",
        F.lit(int(snapshot_date.strftime("%Y%m%d")))
    )
    .withColumnRenamed(
        "salary",
        "salary_amount"
    )
    .select(
        "employee_id",
        "department_id",
        "date_id",
        "salary_amount"
    )
)

display(fact_employee_salary_df)


## 5. Check fact grain and relationships

Check employee/date uniqueness and look for fact rows without matching dimension keys.


In [0]:

print("FACT records:", fact_employee_salary_df.count())

print(
    "Unique employee-date pairs:",
    fact_employee_salary_df
    .select("employee_id", "date_id")
    .distinct()
    .count()
)


In [0]:

missing_employees = (
    fact_employee_salary_df
    .join(dim_employee_df,
            on="employee_id",
            how="left_anti")
    .count()
)

missing_departments = (
    fact_employee_salary_df
    .join(dim_department_df,
            on="department_id",
            how="left_anti")
    .count()
)

missing_dates = (
    fact_employee_salary_df
    .join(dim_date_df,
            on="date_id",
            how="left_anti")
    .count()
)

print("Missing employees:", missing_employees)
print("Missing departments:", missing_departments)
print("Missing dates:", missing_dates)


In [0]:

(
    fact_employee_salary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employees_pipeline.gold.fact_employee_salary")
)

print("FACT_EMPLOYEE_SALARY saved successfully")


## 6. Check saved tables

Read the saved Delta tables and check their row counts.


In [0]:

gold_tables = [
    "dim_employee",
    "dim_department",
    "dim_date",
    "fact_employee_salary"
]

for table in gold_tables:
    df = spark.table(f"employees_pipeline.gold.{table}")
    print(f"{table}: {df.count()}")


## 7. Salary report

Use Spark SQL to calculate employee counts and average salary by department.


In [0]:
%sql


SELECT
    d.department_name,
    COUNT(*) AS employee_count,
    ROUND(AVG(f.salary_amount), 2) AS avg_salary
FROM employees_pipeline.gold.fact_employee_salary f
JOIN employees_pipeline.gold.dim_department d
    ON f.department_id = d.department_id
GROUP BY d.department_name
ORDER BY avg_salary DESC;


## 8. Reconcile Gold with Silver

Compare the salary fact row count with the cleaned Silver data.


In [0]:

report_count = spark.sql("""
    SELECT COUNT(*) AS employee_count
    FROM employees_pipeline.gold.fact_employee_salary
""").first()[0]

silver_count = silver_df.count()

print("Employees in SILVER:", silver_count)
print("Employees in GOLD:", report_count)
print("Balance correct:", report_count == silver_count)
